# Getting started: Git, tokens, big files, splits, leakage and tests

Run this once, before your track's starter notebook, from top to bottom. It
takes about an hour. Every demo uses the SMS Spam Collection, a small text
dataset no team is working on, so nothing here can be pasted into a
deliverable.

Six sections:

1. Git and GitHub from Colab
2. Tokens in Colab Secrets
3. Loading big data without crashing
4. Carving out a validation set
5. A leaky feature
6. Writing a metric against tests

and a checklist at the end.

## 1. Git and GitHub from Colab

Two workflows, and only two:

- **Notebooks** are saved with *File → Save a copy in GitHub*. Colab asks
  which repository and which path; use `notebooks/<yourname>_<topic>.ipynb`.
  That's the whole workflow for notebooks. Never `git add` a notebook from a
  cell; two people doing that on the same file is how merges go wrong.
- **Code in `src/`** is edited in Colab's file browser (the folder icon on
  the left, double-click a file) and pushed with `git` commands in a cell,
  like below.

Before any session: `git pull`, so you're editing the latest code.

The cell below clones the repository. Edit `REPO_URL` first. It reads your
GitHub token from Colab Secrets; section 2 explains how to set that up, so
if the cell fails on the token, read section 2 and come back.

In [ ]:
REPO_URL = "https://github.com/<your-team>/<your-repo>.git"   # <- edit this line

import os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if "<your-team>" in REPO_URL:
        raise ValueError("Edit REPO_URL at the top of this cell to your team's repository first.")
    from google.colab import userdata

    def _secret(name):
        try:
            return userdata.get(name)
        except Exception as e:
            raise RuntimeError(
                f"Colab Secret {name} is missing or not shared with this notebook. "
                "Open the key icon in the left sidebar, add it, and switch on Notebook access."
            ) from e

    token = _secret("GITHUB_TOKEN")
    # This track downloads from UCI, so the Kaggle secrets aren't needed.

    repo = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.isdir(repo):
        auth_url = REPO_URL.replace("https://", f"https://{token}@")
        subprocess.run(["git", "clone", "--quiet", auth_url], check=True)
    os.chdir(repo)
    subprocess.run(["git", "pull", "--quiet"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # Not in Colab: assume this notebook is open from inside the repository.
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

sys.path.insert(0, os.getcwd())
import config
print("repository:", os.getcwd(), "| template version:", config.TEMPLATE_VERSION)
print("TRACK in config.py:", config.TRACK, "" if config.TRACK else "<- one teammate sets this and pushes config.py")

Git needs to know who's committing. These two lines set it for this session
(they're not stored anywhere in the repository):

In [ ]:
MY_NAME = ""        # e.g. "Amira K."
MY_GITHUB = ""      # e.g. "amira-k"

if MY_NAME and MY_GITHUB:
    import subprocess
    subprocess.run(["git", "config", "user.name", MY_NAME], check=True)
    subprocess.run(["git", "config", "user.email", f"{MY_GITHUB}@users.noreply.github.com"], check=True)
    print("git will commit as", MY_NAME)
else:
    print("Fill in MY_NAME and MY_GITHUB above and run this cell again.")

Your first commit: add yourself to `TEAM.md`. This is the pattern for any
change to a file in the repository: edit, `git add`, `git commit`, `git
push`. If the push is rejected, someone else pushed first: `git pull`, then
push again.

In [ ]:
if MY_NAME and MY_GITHUB:
    import subprocess
    with open("TEAM.md", "a") as f:
        f.write(f"| {MY_NAME} | {MY_GITHUB} |\n")
    subprocess.run(["git", "add", "TEAM.md"], check=True)
    subprocess.run(["git", "commit", "-q", "-m", f"Add {MY_NAME} to the team"], check=True)
    r = subprocess.run(["git", "push", "-q"], capture_output=True, text=True)
    print("pushed" if r.returncode == 0 else "push failed:\n" + r.stderr)
else:
    print("Fill in MY_NAME and MY_GITHUB in the cell above first.")

Two things worth knowing about `git` in Colab. The session's disk is wiped
when the runtime disconnects, so anything you haven't pushed is gone with
it: push before you close the tab. And `git status` is cheap; run it when
you're unsure what's changed.

In [ ]:
import subprocess
print(subprocess.run(["git", "status", "--short", "--branch"], capture_output=True, text=True).stdout)

## 2. Tokens in Colab Secrets

Three secrets, created once per Google account, readable from any notebook
you open:

| Name | Where it comes from |
|---|---|
| `GITHUB_TOKEN` | GitHub → Settings → Developer settings → Personal access tokens → Tokens (classic) → Generate new token, with the `repo` scope. Copy it when it's shown; you won't see it again. |
| `KAGGLE_USERNAME` | Your Kaggle username, as in your profile URL. |
| `KAGGLE_KEY` | Kaggle → Settings → API → Create New Token. That downloads a `kaggle.json`; the key is the value inside it. |

To store them: click the key icon in Colab's left sidebar, *Add new
secret*, type the name exactly as above, paste the value, and switch on
*Notebook access*. A cell reads one like this:

In [ ]:
import sys
if "google.colab" in sys.modules:
    from google.colab import userdata
    for name in ("GITHUB_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
        try:
            value = userdata.get(name)
            print(f"{name}: found ({len(value)} characters)")
        except Exception:
            print(f"{name}: missing, or notebook access is off")
else:
    print("Not in Colab; Secrets only exist there.")

**Never paste a token into a cell.** The cell is saved in the notebook, and
the notebook is pushed to GitHub, where everyone with access to the
repository can read it. If the repository is public, GitHub finds its own
tokens and revokes them within minutes, which shows up as pushes suddenly
failing; in a private repository nobody does, and Kaggle keys are never
revoked automatically. If a token is ever committed: revoke it first (GitHub token settings, or Kaggle
→ Settings → API → Expire token), generate a new one, put it in Secrets,
and only then fix the notebook. Removing the cell doesn't remove it from
the history.

## 3. Loading big data without crashing

A free Colab session has about 12 GB of RAM. `pd.read_csv` with default
settings makes every text column an object column (about 60 bytes per
value) and every number a 64-bit one. On a 6-million-row file that's
several GB before you've done anything. Three habits keep it under
control: load only the columns you need (`usecols`), give each one the
smallest dtype that fits, and check with `df.info(memory_usage="deep")`.

The demo file is small, so the numbers below are small, but the ratio is
what to look at. The loader for your track already does this; the point is
to see why.

In [ ]:
import io, zipfile, urllib.request
from pathlib import Path
import pandas as pd

SMS_URL = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
sms_path = Path("data/sms/SMSSpamCollection")

def load_sms():
    """The SMS Spam Collection: one message per line, label then text, tab-separated."""
    if not sms_path.exists():
        sms_path.parent.mkdir(parents=True, exist_ok=True)
        with urllib.request.urlopen(SMS_URL, timeout=60) as r:
            with zipfile.ZipFile(io.BytesIO(r.read())) as z:
                z.extract("SMSSpamCollection", sms_path.parent)
    return pd.read_csv(sms_path, sep="\t", header=None, names=["label", "text"], quoting=3)

sms = load_sms()
print(sms.shape)
print(sms["label"].value_counts())
sms.head()

In [ ]:
# Default load: both columns are plain text.
default = pd.read_csv(sms_path, sep="\t", header=None, names=["label", "text"], quoting=3)
default.info(memory_usage="deep")

In [ ]:
# Same file, label as a category and only the columns we need: compare the "memory usage" line.
lean = pd.read_csv(sms_path, sep="\t", header=None, names=["label", "text"], quoting=3,
                   usecols=["label", "text"], dtype={"label": "category"})
lean.info(memory_usage="deep")

Two more habits. For exploration, work on a sample: `df.sample(100_000,
random_state=42)` is plenty to see the shape of a column, and plots on it
render in seconds. And if a cell runs out of memory, the session dies and
restarts; don't fight it by re-running, reduce what you load.

In [ ]:
sample = sms.sample(500, random_state=42)
print(sample["label"].value_counts(normalize=True).round(3))

## 4. Carving out a validation set

Your track's starter gives you a train pool and a test set. The validation
set is yours to carve from the pool, and there are two patterns.

**Time-based**, when the data has dates or steps: the validation set is the
latest part of the training period. The model is judged on data from after
what it learned from, which is how it will meet real data. The SMS data has
no dates, so the cell below invents a fake day column to show the pattern.

In [ ]:
import numpy as np

demo = sms.copy()
demo["day"] = np.sort(np.random.default_rng(42).integers(0, 30, len(demo)))   # pretend each message has a day

cut = demo["day"].quantile(0.8)                 # the last fifth of the period is validation
train = demo[demo["day"] < cut]
valid = demo[demo["day"] >= cut]

assert train["day"].max() < valid["day"].min(), "training must end before validation begins"
print(f"train: {len(train)} rows (days {train.day.min()}-{train.day.max()}), valid: {len(valid)} rows (days {valid.day.min()}-{valid.day.max()})")

**Stratified random**, when there's no time: a random split that keeps the
share of positives the same in both parts. `train_test_split` does it with
`stratify=`. The check that matters here is that no identifier is in both
parts; the identifier for a text dataset is the text itself.

In [ ]:
from sklearn.model_selection import train_test_split

train, valid = train_test_split(sms, test_size=0.2, stratify=sms["label"], random_state=42)

print("spam share, train: %.3f | valid: %.3f" % ((train.label == "spam").mean(), (valid.label == "spam").mean()))
shared = set(train["text"]) & set(valid["text"])
print("messages in both parts:", len(shared))   # not zero! see section 5

## 5. A leaky feature

That last number wasn't zero. The SMS data has duplicate messages, and a
random split puts copies of the same message on both sides. A model that
memorises them scores on validation as if it had seen the answers, because
it has. The cell below shows it: the same simple model on a split with
duplicates and on one without, and, inside the leaky split, how it does on
the messages it has already seen against the ones it hasn't. The overall
gap is small here because only a few hundred of 5,574 messages repeat; on a
transactions table where the same card appears thousands of times, it isn't.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

def fit(train):
    vec = CountVectorizer(analyzer="char", ngram_range=(3, 3), min_df=2)
    model = LogisticRegression(max_iter=1000).fit(vec.fit_transform(train["text"]), train["label"] == "spam")
    return vec, model

def score(vec, model, valid):
    return model.score(vec.transform(valid["text"]), valid["label"] == "spam")

leaky_train, leaky_valid = train_test_split(sms, test_size=0.2, stratify=sms["label"], random_state=42)
dedup = sms.drop_duplicates("text")
clean_train, clean_valid = train_test_split(dedup, test_size=0.2, stratify=dedup["label"], random_state=42)

print("duplicate messages in the file:", len(sms) - len(dedup))
vec, model = fit(leaky_train)
print("accuracy with duplicates across the split: %.4f" % score(vec, model, leaky_valid))
seen = leaky_valid["text"].isin(set(leaky_train["text"]))
for label, part in (("also in training", leaky_valid[seen]), ("not in training ", leaky_valid[~seen])):
    if len(part):
        print("   on validation messages %s: %.4f  (%d messages)" % (label, score(vec, model, part), len(part)))
vec, model = fit(clean_train)
print("accuracy after de-duplicating first:       %.4f" % score(vec, model, clean_valid))

The second kind of leak is a feature that looks at the future. On a table
of transactions, "how many transactions this account made in the *past*
hour" is fine: at prediction time, the past has happened. "In the *next*
hour" is not: it won't exist when the model has to decide, and in the
training data it's often correlated with the label (a fraudster's burst
shows up as a high next-hour count on the first transaction). The tiny
table below makes both, so you can see that the leaky one is just as easy
to write.

In [ ]:
tx = pd.DataFrame({
    "account": ["A", "A", "A", "B", "B", "A", "B"],
    "minute":  [0,   10,  20,  5,   50,  65,  70 ],
    "amount":  [20,  15,  500, 30,  25,  12,  600],
})
tx = tx.sort_values(["account", "minute"]).reset_index(drop=True)

def count_in_window(df, before: bool):
    out = []
    for _, row in df.iterrows():
        same = df[df["account"] == row["account"]]
        if before:   # strictly earlier, within the last 60 minutes: fine
            n = ((same["minute"] < row["minute"]) & (same["minute"] >= row["minute"] - 60)).sum()
        else:        # strictly later, within the next 60 minutes: leaks
            n = ((same["minute"] > row["minute"]) & (same["minute"] <= row["minute"] + 60)).sum()
        out.append(int(n))
    return out

tx["past_hour_count"] = count_in_window(tx, before=True)    # OK
tx["next_hour_count"] = count_in_window(tx, before=False)   # LEAKS
tx

The rule: a feature for a row may use only what existed at that row's
time. Also quieter and just as bad: a statistic computed over the whole
file (a merchant's average amount including test rows) is legal to know at
prediction time but was computed from rows you promised not to touch.
Compute such things on the training pool only.

## 6. Writing a metric against tests

Your track's metric is written to pass a test file with known answers. The
rhythm is: run the test and watch it fail, write the function, run again,
until green. Here it is on a metric no track uses, balanced accuracy at a
threshold of 0.5, so the pattern is clear before you do it for real.

First, the test. It says what the function must return on inputs where the
answer is known by hand, including a case with no positives at all.

In [ ]:
test_source = '''
import math
import numpy as np
from toy_metric import balanced_accuracy

def test_perfect():
    assert balanced_accuracy([0, 0, 1, 1], [0.1, 0.2, 0.9, 0.8]) == 1.0

def test_half_right_on_each_class():
    # one of two negatives right, one of two positives right -> (0.5 + 0.5) / 2
    assert balanced_accuracy([0, 0, 1, 1], [0.1, 0.9, 0.9, 0.1]) == 0.5

def test_ties_at_the_threshold_count_as_flagged():
    # score >= 0.5 is flagged, so 0.5 on a positive is a hit
    assert balanced_accuracy([0, 1], [0.1, 0.5]) == 1.0

def test_no_positives():
    assert math.isnan(balanced_accuracy([0, 0], [0.1, 0.2]))
'''
open("test_toy_metric.py", "w").write(test_source)

# A stub, like the ones in src/metrics.py: it raises until written.
open("toy_metric.py", "w").write('''
def balanced_accuracy(y_true, scores):
    """Mean of (true positive rate, true negative rate) at threshold 0.5. nan if a class is missing."""
    raise NotImplementedError("write me")
''')

!python -m pytest -q test_toy_metric.py

Four failures, each naming the test. Now the implementation. Read the
docstring and the tests, write the smallest thing that satisfies them:

In [ ]:
open("toy_metric.py", "w").write('''
import numpy as np

def balanced_accuracy(y_true, scores):
    """Mean of (true positive rate, true negative rate) at threshold 0.5. nan if a class is missing."""
    y = np.asarray(y_true).astype(int)
    flagged = np.asarray(scores, dtype=float) >= 0.5
    pos, neg = y == 1, y == 0
    if pos.sum() == 0 or neg.sum() == 0:
        return float("nan")
    tpr = flagged[pos].mean()
    tnr = (~flagged[neg]).mean()
    return float((tpr + tnr) / 2)
''')

!python -m pytest -q test_toy_metric.py

Green. The two cases that catch people are the ones at the end: a tie on
the threshold, and a slice with nothing to detect. Your track's tests have
the same two kinds of case, for the same reason. When you write your
metric, start from its docstring in `src/metrics.py`, which spells out ties
and empties, and from the failing test's message, which names the case.

Cleaning up the demo files:

In [ ]:
import os
for f in ("toy_metric.py", "test_toy_metric.py"):
    if os.path.exists(f):
        os.remove(f)

## Checklist

Before you open your track's starter notebook:

- [ ] One teammate created the repository from the template and invited
      the others, the TPM, Ben and Pierre.
- [ ] You've committed once, from this notebook, and your line is in `TEAM.md`.
- [ ] `GITHUB_TOKEN`, `KAGGLE_USERNAME` and `KAGGLE_KEY` are in your Colab Secrets.
- [ ] `TRACK` is set in `config.py` and pushed (one teammate).
- [ ] Your track's starter runs through to the test split.
- [ ] `pytest -q` runs. The metric tests fail; that's expected until D1.5.